# Re-Ranking — Cross-Encoder for Precision

## Bi-Encoder vs Cross-Encoder

**Bi-encoder** (used in semantic search):
- Embeds query and document SEPARATELY
- Fast: can pre-compute document embeddings
- But: never sees query+doc together, misses nuanced relevance

**Cross-encoder** (used for re-ranking):
- Sees query AND document TOGETHER as input
- Slow: must process each (query, doc) pair individually
- But: much more accurate relevance scoring

**Strategy:** Use bi-encoder for fast initial retrieval (top 20), then cross-encoder to re-rank for precision (top 5).

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

# Load both models
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

print("Bi-encoder: all-MiniLM-L6-v2 (embeds separately, fast)")
print("Cross-encoder: ms-marco-MiniLM-L6-v2 (sees both together, accurate)")

In [ ]:
query = "Side effects of metformin"

docs = [
    "Metformin 500mg is the first-line treatment for type 2 diabetes. Common side effects include nausea, diarrhea, stomach pain, and a metallic taste.",
    "Type 2 diabetes treatment algorithm: Step 1 — Lifestyle modifications. Step 2 — Metformin monotherapy.",
    "Atorvastatin 20mg side effects include muscle pain, liver enzyme elevation, and rarely rhabdomyolysis.",
    "Lisinopril 10mg side effects include dry cough, dizziness, and hyperkalemia.",
    "Insulin glargine is a long-acting basal insulin for diabetes management. Hypoglycemia is the main risk.",
]

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# Stage 1: Bi-encoder ranking (fast initial retrieval)
query_emb = bi_encoder.encode(query)
doc_embs = bi_encoder.encode(docs)

bi_scores = [cosine_sim(query_emb, d) for d in doc_embs]
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)

print("=== Stage 1: Bi-Encoder Ranking (Semantic Search) ===")
print(f"Query: \"{query}\"\n")
for rank, (idx, score) in enumerate(bi_ranked, 1):
    print(f"  #{rank} ({score:.4f}): {docs[idx][:70]}...")

In [ ]:
# Stage 2: Cross-encoder re-ranking
pairs = [(query, doc) for doc in docs]
ce_scores = cross_encoder.predict(pairs)

ce_ranked = sorted(enumerate(ce_scores), key=lambda x: x[1], reverse=True)

print("=== Stage 2: Cross-Encoder Re-Ranking ===")
print(f"Query: \"{query}\"\n")
for rank, (idx, score) in enumerate(ce_ranked, 1):
    # Find original bi-encoder rank
    original_rank = next(r for r, (i, _) in enumerate(bi_ranked, 1) if i == idx)
    moved = original_rank - rank
    arrow = f"(↑{moved})" if moved > 0 else f"(↓{-moved})" if moved < 0 else "(=)"
    print(f"  #{rank} ({score:.4f}) {arrow}: {docs[idx][:70]}...")

In [ ]:
# Speed comparison
import time

# Bi-encoder: encode once, compare many
start = time.perf_counter()
for _ in range(100):
    q = bi_encoder.encode(query)
    [cosine_sim(q, d) for d in doc_embs]
bi_time = (time.perf_counter() - start) / 100 * 1000

# Cross-encoder: process each pair
start = time.perf_counter()
for _ in range(100):
    cross_encoder.predict(pairs)
ce_time = (time.perf_counter() - start) / 100 * 1000

print(f"Bi-encoder avg:    {bi_time:.1f}ms for {len(docs)} docs")
print(f"Cross-encoder avg: {ce_time:.1f}ms for {len(docs)} docs")
print(f"Cross-encoder is {ce_time/bi_time:.1f}x slower")
print(f"\nThat's why we re-rank only top-20, not the entire corpus!")

## Key Takeaways

1. **Bi-encoder is fast but approximate** — embeds query and docs separately
2. **Cross-encoder is slow but precise** — sees query+doc together
3. **Two-stage pipeline** = best of both worlds: speed + accuracy
4. **Only re-rank a small candidate set** (top 20) — cross-encoder is too slow for full corpus
5. **Re-ranking often reshuffles results** — the #3 bi-encoder result might become #1 after cross-encoder